# **Infer `Golgi`**

***Prior to this notebook, you should have already run through [1.0_image_setup](1.0_image_setup.ipynb).***

<mark> TO DO EVENTUALLY: Include more information on what the expect output is for each step; include a figure in the intro section with expected output from the sample data file. Update bio relevance with full citations.</mark>

In notebooks 1.1 through 1.8, we will go over the implementation of `infer-subc` in the a Napari plugin called  `organelle-segmenter-plugin`. The steps outlined in each notebook correlate to the workflow steps from the plugin. Here, you will gain knowledge of what `infer-subc` functions are involved in each of those steps.

The segmentation workflows are completely independent of each other and can be run in any order. The current notebook segments the `Golgi`. Notesbook for the segmentation of other organelles can be found here:
- Infer [`lysosomes`](1.2_infer_lysosome.ipynb)
- Infer [`mitochondria`](1.3_infer_mitochondria.ipynb)
- Infer [`peroxisomes`](1.5_infer_peroxisome.ipynb)
- Infer [`endoplasmic reticulum (ER)`](1.6_infer_ER.ipynb)
- Infer [`lipid droplets`](1.7_infer_lipid_droplet.ipynb)

***Note:** These notebooks do not include a batch processing option. They are simply explanations of the steps included in the Napari plugin. If you would like to batch process your images in a Jupyter notebook instead of the plugin, you can use the [batch process segmentation](batch_process_segmentations.ipynb) notebook.*

### **Single cell analysis**
Each workflow segments organelles from the entire image, irrespective of the cell mask identified in notebook 1.1. To attain a single-cell analysis, the mask will be applied to the organelle segmentation outputs before quantification in the `part_2_quantification` notebooks. 

### **Golgi** 📦✉️
The Golgi is a membrane-enclosed complex that is part of the secretory pathway. Proteins and lipids are synthesized in the endoplasmic reticulum (ER), transferred to the Golgi where they are modified as needed, and then packaged into vesicles for further transport throughout the cell. The Golgi also acts as a signaling hub for nutrient sensing, cell growth and metabolism, and is involved in organelle contacts with other membrane-bound organelles. 

### **Fluorescence labeling strategies** 🔆 
The Golgi can be fluorescently labeled in live and fixed cells by staining for endogenous proteins, targeting fluorescent proteins to the membrane or lumen using genertically encoded markers, or with dye-based approaches. The labeling approach used and resolution of your images can result in different staining outcomes that can impact the segmentation steps below. Additionally, sub-compartments within the Golgi have differing resident proteins making genetic and immunolabeling approaches nuanced. For example, labeling the GM130 protein will target the cis-Golgi while p230 will target the trans-Golgi network. These structures will likely have slightly different characteristics.

This workflow was optimized for images of fluorescently tagged sialyltransferase 1 (SiT1), a Golgi membrane protein. The resulting Golgi structure has both the classical cisternae as well as small Golgi-derived vesicles; so we are utilizing two segmentation methods to idenfity the entire Golgi structure.

***We advise that you test the segmentation process on a small, pilot dataset before committing to a particular labeling strategy and segmentation approach.***

-----

### 👣 **Summary of steps**  

➡️ **EXTRACTION**
- **`STEP 1`** - Select a channel for segmentation

    - select single channel containing the golgi marker (channel number = user input)

**PRE-PROCESSING**
- **`STEP 2`** - Rescale and smooth image

  - rescale intensity of composite image (min=0, max=1)
  - median filter (median size = user input)
  - gaussian filter (sigma = user input)

**CORE PROCESSING**
- **`STEP 3`** - Global + local thresholding (AICSSeg – MO)

    - apply MO thresholding method from the Allen Cell [aicssegmentation](https://github.com/AllenCell/aics-segmentation) package (threshold options = user input)

- **`STEP 4`** - Thin segmentation (Topology preserving)

    - apply topology preserved thinning function from the Allen Cell [aicssegmentation](https://github.com/AllenCell/aics-segmentation) package (thinning amount, minimum thickness = user input)

- **`STEP 5`** - ‘Dot’ thresholding method (AICSSeg)

    - apply "dot" thresholding method (for small round objects) from the Allen Cell [aicssegmentation](https://github.com/AllenCell/aics-segmentation) package (size scale and threshold cutoff = user input)

- **`STEP 6`** - Combine Segmentations (logical or)

    - combine the two segmentations with logical *OR*

**POST-PROCESSING**
- **`STEP 7`** - Remove small holes and objects

  - fill holes (hole size = user input)
  - remove small objects (object size = user input)
  - filter method (method = user input)

**POST-POST-PROCESSING**
- **`STEP 8`** - Label objects

  - label unique golgi objects based on connectivity or declumping

**EXPORT** ➡️
- save labeled ***golgi*** (golgi, GL) as unsigned integer 16-bit tif files


*This workflow is an adaptation of the Allen Cell Segmenter procedure for segmentation of Golgi from the [sialyltransferase 1 (ST6GAL1)](https://www.allencell.org/cell-observations/category/golgi-apparatus) marker. Sourced from: this [script](https://github.com/AllenCell/aics-segmentation/blob/main/aicssegmentation/structure_wrapper/seg_st6gal1.py).*

---------------------
## **IMPORTS AND LOAD IMAGE**

Details about the functions included in this subsection are outlined in the [`1.0_image_setup`](1.0_image_setup.ipynb) notebook. Please visit that notebook first if you are confused about any of the code included here.

#### &#x1F3C3; **Run code; no user input required**

In [1]:
from pathlib import Path
import os

import numpy as np
import pandas as pd
import napari
from napari.utils.notebook_display import nbscreenshot

from aicssegmentation.core.utils import topology_preserving_thinning

from infer_subc.core.file_io import (read_czi_image,
                                     export_inferred_organelle,
                                     list_image_files,
                                     sample_input)
from infer_subc.core.img import *
# from infer_subc.organelles.declumping import watershed_declumping

from bioio import BioImage

%load_ext autoreload
%autoreload 2

#### &#x1F6D1; &#x270D; **User Input Required:**

Please specify the following information about your data: 
- `im_type`: the file type of your image written in quotation marks; this must be written within quotation markers. 
    - EX: ".czi" or ".tiff"
- `data_root_path`: the path to folder that your input data and a separate folder for segmentation outputs; this must be written within quotation markers. If you are using a Windows computer, paths are copied with backslashes ("\"). Python will be confused by this; switch these to forward slashes ("/") or place an "r" in front of the path string.  
    - EX: "C:/Users/{your-user-name}/Documents/Exp1" or r"C:\Users\{your-user-name}\Documents\Exp1"
- `raw_data_path`: the path to the folder that contains your input data; this must be written within quotation markers. 
    - EX: data_root_path / "input"
- `seg_data_path`: the path to the folder where segmentation output files will be saved; this must be written within quotation markers. EX: data_root_path / "segmentations". If this folder does not exist already, it will be created for you in the subsequent lines of code.

In [2]:
#### USER INPUT REQUIRED ###
im_type = ".czi"
data_root_path = Path(os.path.dirname(os.getcwd())) 
raw_data_path = data_root_path / "pilot-data"
seg_data_path = data_root_path / "test-segmentation"

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** A list of the images included in the `raw_data_path` folder is printed below for easy reference below.

In [3]:
# Create the output directory to save the segmentation outputs in.
if not Path.exists(seg_data_path):
    Path.mkdir(seg_data_path)
    print(f"Segmentation data path did not exist; making it now: {seg_data_path}")

# list files in the input folder
print(f"\nListing all {im_type} files in the raw data path: ")
img_file_list = list_image_files(Path(raw_data_path),im_type)
pd.set_option('display.max_colwidth', None)
pd.DataFrame({"Image Name":img_file_list})


Listing all .czi files in the raw data path: 


,Image Name
0,c:\Users\Shannon\Documents\Python_Scripts\Golgi-coloc-analysis\pilot-data\25-02-25 SLC35A2 WT -01.czi
1,c:\Users\Shannon\Documents\Python_Scripts\Golgi-coloc-analysis\pilot-data\25-07-25 SLC35A2 R55L-01.czi
2,c:\Users\Shannon\Documents\Python_Scripts\Golgi-coloc-analysis\pilot-data\25-07-25 SLC35A2 S304P-01.czi
3,c:\Users\Shannon\Documents\Python_Scripts\Golgi-coloc-analysis\pilot-data\25-07-25 SLC35A2 Val206Phe-01.czi


#### &#x1F6D1; &#x270D; **User Input Required:**

Use the list above to specify which image you wish to analyze based on its index: 
- `test_img_n`: the index, or number, associated with your image of choice. This value can be found to the left of the image path listed in the table above.
- `channel_names`: specify the name of each channel in your image in order. The name of each channel should be within quotation markers and separated by a common. The entire list should be within square brackets ([ ]) -- a Python list.
    - EX: ['DAPI', 'slc35a2', 'golgi']

In [4]:
#### USER INPUT REQUIRED ###
test_img_n = 0
channel_names = ['golgi', 'slc35a2', 'DAPI']

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** This code block reads the image and image metadata into memory. The metadata is printed and the image is visualized in Napari. A new window will appear displaying the Napari Viewer.

<mark> **THIS METHOD ONLY READ THE FIRST IMAGE IN EVERY SERIES - need to fix this to read each one separately**

In [5]:
# read in image and metadata
file_path = str(img_file_list[test_img_n])
raw_file = BioImage(file_path)
img_data = np.squeeze(raw_file.data) # np.squeeze removes single-dimensional entries from the shape of an array
meta_dict = raw_file.standard_metadata

# extra metadata information
dimensions = meta_dict.dimensions_present
channel_axis = str(dimensions).index('C')
dim_size = {"Z": raw_file.physical_pixel_sizes.Z, 
            "Y": raw_file.physical_pixel_sizes.Y, 
            "X": raw_file.physical_pixel_sizes.X}
# select the scale values from the dim_size dict in the order in which X, Y, and Z appear in the dimensions string
scale_order = [dim for dim in str(dimensions) if dim in ['Z','Y','X']]
scale = tuple([dim_size[dim] for dim in scale_order])

print("Metadata information")
print(f"File path: {file_path}")
for i in list(range(len(channel_names))):
    print(f"Channel {i} name: {channel_names[i]}")
print(f"Scale ({scale_order}): {scale}")
print(f"Channel axis: {channel_axis}")

# open viewer and add images
viewer = napari.Viewer()
for i, channel_name in enumerate(channel_names):
    viewer.add_image(img_data[i],
                     scale=scale,
                     name=f"Channel {i}: {channel_name}")
viewer.grid.enabled = True
viewer.reset_view()
print("\nProceed to Napari window to view your selected image.")

Metadata information
File path: c:\Users\Shannon\Documents\Python_Scripts\Golgi-coloc-analysis\pilot-data\25-02-25 SLC35A2 WT -01.czi
Channel 0 name: golgi
Channel 1 name: slc35a2
Channel 2 name: DAPI
Scale (['Z', 'Y', 'X']): (0.2, 0.07059082892416223, 0.07059082892416223)
Channel axis: 0


c:\Users\Shannon\anaconda3\envs\golgi-coloc\lib\site-packages\npe2\manifest\_npe1_adapter.py:87: UserWarning: Error importing contributions for first-generation napari plugin 'napari-aicsimageio': cannot import name 'Dimensions' from 'aicsimageio.constants' (c:\Users\Shannon\anaconda3\envs\golgi-coloc\lib\site-packages\aicsimageio\constants.py)
  self._load_contributions()



Proceed to Napari window to view your selected image.


-----
## **EXTRACTION**

### **`STEP 1` - Select a channel for segmentation**

#### &#x1F6D1; &#x270D; **User Input Required:**

Please specify which channel includes your Golgi label:
- `GOLGI_CH`: the index of the channel containing your Golgi label. Image indexing begins with 0, not 1. Reference the channel numbers indicated in the Napari window for easy reference.

> ⚙️ **SAMPLE DATA SETTINGS**
> 
> If using the sample data, here are the correct channel indices
> 
> |**Cell type**|GOLGI_CH| 
> | :------------------------------------- |  :------:  |
> |neuron_1|`2`|
> |astrocyte|`2`|
> |neuron_2|`2`|
> |iPSC|`4`|

In [6]:
#### USER INPUT REQUIRED ###
GOLGI_CH = channel_names.index('golgi')

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** This code block extracts the Golgi channel from your multi- (or single) channel image. It will be the only part of the image used in the rest of this workflow. The single golgi channel is added to the Napari viewer.

In [7]:
# select channel
raw_golgi = select_channel_from_raw(img_data, GOLGI_CH)

# clear napari and add single channel as a new layer
viewer.layers.clear()
viewer.grid.enabled = False
viewer.add_image(raw_golgi, scale=scale, name="1 - Extract Golgi")

<Image layer '1 - Extract Golgi' at 0x1e3ca7ec8e0>

-----
## **PRE-PROCESSING**

### **`STEP 2` - Rescale and smooth image**

&#x1F453; **FYI:** This code block rescales the image so that the pixel/voxel with the highest intensity is set to 1 and the one with the lowest intensity is set to 0. The image is then *optionally* smoothed using a Gaussian and/or median filter. 

<mark> Include more information on the Gaussian and median filtering methods here </mark>

#### &#x1F6D1; &#x270D; **User Input Required:**

Please specify the amount of filter to use for each method. Higher values indicate more smoothing:
- `med_filter_size`: the size of the median filter to apply; if 0 is used, no filter will be applied
- `gaussian_smoothing_sigma`: the sigma to apply in the Gaussian filtering step; if 0 is used, no filter will be applied

> ⚙️ **SAMPLE DATA SETTINGS**
> 
> If using the sample data, here are the recommended parameters for each cell type
> 
> |**Cell type**|med_filter_size|gaussian_smoothing_sigma| 
> | :------------------------------------- |  :------:  |  :------:  |
> |neuron_1|`0`|`0.0`|
> |astrocyte|`0`|`0.0`|
> |neuron_2|`0`|`0.0`|
> |iPSC|`0`|`0.0`|

In [8]:
#### USER INPUT REQUIRED ###
median_sz = 4
gauss_sig = 1.4

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** This code block rescales the image and applies the specified median and Gaussian filters. The image is then added to Napari as a new layer for visual comparison to the input image. 

Use the Napari viewer to iteratively adjust the smoothing settings selected above.

In [9]:
# rescaling and smoothing input image
struct_img =  scale_and_smooth(raw_golgi,
                               median_size = median_sz, 
                               gauss_sigma = gauss_sig)

# adding image to Napari as a new layer
viewer.add_image(struct_img, scale=scale, name=f"2 - Rescale and Smooth (median_sz={median_sz}, gauss_sig={gauss_sig})")

<Image layer '2 - Rescale and Smooth (median_sz=4, gauss_sig=1.4)' at 0x1e3c7e5c190>

-----
## **CORE-PROCESSING**

### **`STEP 3` - Global + local thresholding (AICSSeg – MO)**


&#x1F453; **FYI:** This code block is the first of two semantic segmentation steps that are combined together in a later step, and is used to segment Golgi cisterae strucutres. `Semantic segmentation` is the process of deciding whether a pixel/voxel should be included in an object (labeled with a value of 1) or should be considered as part of the background (labeled with a value of 0). A semantic segmentation does not discern individual objects from one another.

The masked_object_filter utilizes the 'MO' filter from the [`aics-segmentation`](https://github.com/AllenCell/aics-segmentation) package. AICS documentation states: "The algorithm is a hybrid thresholding method combining two levels of thresholds. The steps are: [1] a global threshold is calculated, [2] extract each individual connected componet after applying the global threshold, [3] remove small objects, [4] within each remaining object, a local Otsu threshold is calculated and applied with an optional local threshold adjustment ratio (to make the segmentation more and less conservative). An extra check can be used in step [4], which requires the local Otsu threshold larger than 1/3 of global Otsu threhsold and otherwise this connected component is discarded."

#### &#x1F6D1; &#x270D; **User Input Required:**

Please specify the following values:
- `thresh_method`: the global thresholding method; options include 'tri'/'triangle, 'med'/'median' or 'ave'/'ave_tri_med'. Triangle implements the [skimage.filters.threshold_triangle](https://scikit-image.org/docs/stable/api/skimage.filters.html#skimage.filters.threshold_triangle) method, median utilizes the 50th percentile of the intensities, and ave uses the average of the two methods to calculate the lower bound of the global threshold.
- `cell_wise_min_area`: the minimum expected size of your object; smaller objects will be removed prior to local thresholding
- `thresh_adj`: adjustment to make to the local threshold; larger values make the segmentation more stringent (less area included)

> ⚙️ **SAMPLE DATA SETTINGS**
> 
> If using the sample data, here are the recommended parameters for each cell type
> 
> |**Cell type**|thresh_method|cell_wise_min_area|thresh_adj|
> | :------------------------------------- |  :------:  |  :------:  |  :------:  |
> |neuron_1|`'ave_tri_med'`|`1200`|`.9`|
> |astrocyte|`'ave_tri_med'`|`1200`|`.9`|
> |neuron_2|`'triangle'`|`1200`|`.8`|
> |iPSC|`'ave_tri_med'`|`1200`|`.6`|

In [10]:
#### USER INPUT REQUIRED ###
thresh_method = 'median'
cell_wise_min_area = 1200
thresh_adj = 1.3

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** This code block executes the MO filter using the settings above.

Use the Napari viewer to iteratively adjust the filter settings as needed.

In [11]:
# segment the majority of the golgi with this global and local thresholding method
bw = masked_object_thresh(struct_img, 
                          global_method=thresh_method, 
                          cutoff_size=cell_wise_min_area, 
                          local_adjust=thresh_adj)

# adding image to Napari as a new layer
viewer.add_image(bw, scale=scale, name=f"3 - MO filter (thresh_method={thresh_method}, thresh_adj={thresh_adj}, size={cell_wise_min_area})", opacity=0.3, colormap="cyan", blending='additive')

<Image layer '3 - MO filter (thresh_method=median, thresh_adj=1.3, size=1200)' at 0x1e3ca7b6110>

### **`STEP 4` - Thin segmentation (topology preserving)**

&#x1F453; **FYI:** This code block implements a thinning step to refine the result of the MO segmentation. This logic was developed in the [`aics-segmentation`](https://github.com/AllenCell/aics-segmentation) package. The [topology_preserve_thinning](https://github.com/AllenCell/aics-segmentation/blob/main/aicssegmentation/core/utils.py#L101) function reduces the diameter of the segmented object while preserving its overall topology.

#### &#x1F6D1; &#x270D; **User Input Required:**

Please specify the following values:
- `thin_dist_preserve`: half of the minimum width of your object
- `thin_dist`: the amount to thin by in pixels/voxels; must be a positive integer; a value of 0 can be used to if no thinning is needed

> ⚙️ **SAMPLE DATA SETTINGS**
> 
> If using the sample data, here are the recommended parameters for each cell type
> 
> |**Cell type**|thin_dist_preserve|thin_dist|
> | :------------------------------------- |  :------:  |  :------:  |
> |neuron_1|`8`|`1`|
> |astrocyte|`8`|`1`|
> |neuron_2|`1`|`1`|
> |iPSC|`4`|`2`|

In [12]:
#### USER INPUT REQUIRED ###
thin_dist_preserve = 5
thin_dist = 2

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** This code block executes the thinning step using the settings above.

Use the Napari viewer to iteratively adjust the filter settings as needed.

In [34]:
# thin segmentation with maintain a minimal required thickness
bw_thin = topology_preserving_thinning(bw, thin_dist_preserve, thin_dist)

# adding image to Napari as a new layer
viewer.add_image(bw_thin, scale=scale, name=f"4 - Thinning (preserve={thin_dist_preserve}, thin_dist={thin_dist})", opacity=0.3, colormap="magenta", blending='additive')

<Image layer '4 - Thinning (preserve=5, thin_dist=2) [1]' at 0x1e3ccb85240>

### **`STEP 5` -  ‘Dot’ thresholding method (AICSSeg)**

&#x1F453; **FYI:** This code block is the second of two semantic segmentation steps that are combined together in a later step, and is used to segment Golgi-derived vesicles. The 'dot' filter is derived from the [`aics-segmentation`](https://github.com/AllenCell/aics-segmentation) package. It utilizes up to three scale (object size) and cutoff (threshold) pairs for objects of different size and intensity. This function is specifically designed to segment small round objects.

#### &#x1F6D1; &#x270D; **User Input Required:**

Please specify the scale and cutoff values for each pair:
- `dot_scale_1`: the size scale for the first scale/cutoff pair; larger values correlate to selection of larger objects
- `dot_cut_1`: the threshold cutoff value for the first scale/cutoff pair; small cutoffs tend to yield more dots that are larger in volume; larger cutoffs tend to yield less dots that are slimmer.
- `dot_scale_2`: the size scale for the second scale/cutoff pair; this can be set to 0 if a second scale/cutoff pair isn't needed
- `dot_cut_2`: the threshold cutoff value for the second scale/cutoff pair; this can be set to 0 if a second scale/cutoff pair isn't needed
- `dot_scale_3`: the size scale for the third scale/cutoff pair; this can be set to 0 if a third scale/cutoff pair isn't needed
- `dot_cut_3`: the threshold cutoff value for the third scale/cutoff pair; this can be set to 0 if a third scale/cutoff pair isn't needed
- `dot_method`: "3D" processes the image taking into account intensities in three dimensions (XYZ); "slice-by-slice" processes each Z-slice in the image separately, not considering any intensity in higher or lower Z planes.

> ⚙️ **SAMPLE DATA SETTINGS**
> 
> If using the sample data, here are the recommended parameters for each cell type
> 
> |**Cell type**|dot_scale_1|dot_cut_1|dot_scale_2|dot_cut_2|dot_scale_3|dot_cut_3|dot_method|
> | :------------------------------------- |  :------:  |  :------:  |  :------:  |  :------:  |  :------:  |  :------:  |  :------:  |
> |neuron_1|`0.5`|`0.1`|`0`|`0`|`0`|`0`|`"3D"`|
> |astrocyte|`0.5`|`0.1`|`0`|`0`|`0`|`0`|`"3D"`|
> |neuron_2|`1.2`|`0.03`|`0`|`0`|`0`|`0`|`"3D"`|
> |iPSC|`1.6`|`0.04`|`0`|`0`|`0`|`0`|`"slice_by_slice"`|

In [14]:
#### USER INPUT REQUIRED ###
dot_scale_1 = 0.05
dot_cut_1 = 0.65

dot_scale_2 = 0
dot_cut_2 = 0

dot_scale_3 = 0
dot_cut_3 = 0

dot_method = '3D'

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** This code block executes the dot filter using the settings above.

Use the Napari viewer to iteratively adjust the filter settings as needed. 

*Hint: it is helpful to adjust the scale/cutoff filters one at a time and then combine them once each pair's settings are confirmed.*

In [15]:
# segment small round structures with this (golgi vesicles)
bw_extra = dot_filter_3(struct_img,
                        dot_scale_1,
                        dot_cut_1,
                        dot_scale_2,
                        dot_cut_2,
                        dot_scale_3,
                        dot_cut_3,
                        dot_method)

# adding image to Napari as a new layer
viewer.add_image(bw_extra, scale=scale, name=f"5 - Dot filter (scale/cutoff 1 = {dot_scale_1}/{dot_cut_1}, scale/cutoff 2 = {dot_scale_2}/{dot_cut_2}, scale/cutoff 3 = {dot_scale_3}/{dot_cut_3})", opacity=0.3, colormap="green", blending='additive')

<Image layer '5 - Dot filter (scale/cutoff 1 = 0.05/0.65, scale/cutoff 2 = 0/0, scale/cutoff 3 = 0/0)' at 0x1e3ca7b4820>

### **`STEP 6` - Combine Segmentations**

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** This code block combines the results of the thinning and the dot filter together using the "logical or" operation. The resulting semantic segmentation can be viewed in Napari.

In [35]:
# combine the two segmentations together
bw_combine = np.logical_or(bw_extra, bw_thin)

# adding image to Napari as a new layer
viewer.add_image(bw_combine, scale=scale, name="6 - Combined Semantic Segmentation", opacity=0.3, blending='additive')

<Image layer '6 - Combined Semantic Segmentation [1]' at 0x1e3ccb86c50>

-----
## **POST-PROCESSING**

### **`STEP 7` - Remove small holes and objects**

&#x1F453; **FYI:** This code block cleans up the semantic segmentation by filling small holes and/or removing small objects that can be considered errors in the initial segmentation. 

#### &#x1F6D1; &#x270D; **User Input Required:**

Please specify the following values:
- `hole_min_width`: the width of the smallest hole to be filled
- `hole_max_width`: the width of the largest hole to be filled
- `small_object_width`: the width of the largest object to be removed; any object smaller than this size will be removed
- `fill_filter_method`: "3D" processes the image taking into account segmentation values in three dimensions (XYZ); "slice-by-slice" processes each Z-slice in the image separately, not considering the segmentation results in higher or lower Z planes.

> ⚙️ **SAMPLE DATA SETTINGS**
> 
> If using the sample data, here are the recommended parameters for each cell type
> 
> |**Cell type**|hole_min_width|hole_max_width|small_object_width|fill_filter_method|
> | :------------------------------------- |  :------:  |  :------:  |  :------:  |  :------:  |
> |neuron_1|`0`|`0`|`2`|`"3D"`|
> |astrocyte|`0`|`0`|`2`|`"3D"`|
> |neuron_2|`0`|`4`|`2`|`"3D"`|
> |iPSC|`0`|`0`|`4`|`"3D"`|

In [17]:
#### USER INPUT REQUIRED ###
hole_min_width = 0
hole_max_width = 0

small_object_width = 5

fill_filter_method = "3D"

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** This code block fills small holes and removes small objects using the settings above.

Use the Napari viewer to iteratively adjust the filter settings as needed. 

*Hint: white pixels/voxels are the ones remaining after this step*

In [36]:
# fill holes and removed small objects
cleaned_img2 = fill_and_filter_linear_size(bw_combine, 
                                           hole_min=hole_min_width, 
                                           hole_max=hole_max_width, 
                                           min_size=small_object_width,
                                           method=fill_filter_method)

# adding image to Napari as a new layer
viewer.add_image(cleaned_img2, scale=scale, name="7 - Fill holes and remove small objects", colormap="magenta", blending="additive")

<Image layer '7 - Fill holes and remove small objects [1]' at 0x1e3e58e3670>

-----
## **POST-POST-PROCESSING**

### **`STEP 8` - Label objects**

<mark> **Declumping function had to be added becuase they are only available in infer-subc v2b; currently using infer-subc v1 implementation)**

&#x1F453; **FYI:** This code block takes the semantic segmentation and creates an `instance segmentation`. In this output, each individual object in the image is given a unique ID number. The background pixels/voxels are still labeled as 0, but now each pixel/voxel within an object is labeled as a positive integer matching the ID number for that object. 

In this workflow objects may be separated based on `connectivity`: if a pixel/voxel is touching another pixel/voxel in any direction, they are considered the same object and each pixel/voxel within that object is labeled as the same unique ID number. Alternatively, workflow objects may be separated based on a `declumping` method—detailed in the [method_declumping](method_declumping.ipynb)—thus allowing tightly packed objects to be labeled as separate objects. Caveats to this method are also detailed in the [method_declumping](method_declumping.ipynb). 

#### &#x1F6D1; &#x270D; **User Input Required:**

Please specify the following values:
- `declump`: A True/False statement of whether to use the declumping method or not.
- `sigma`: A float value to apply in the gaussian filtering of the highpass filter. (Ignored if declump is False)
- `iterations`: The number of iterations to apply to the highpass filter. (Ignored if declump is False)
- `open_img`: A True/False statement of whether to open the highpass filtered image or not. (Ignored if declump is False)
- `thresh_adj`: A scalar to the automatically determined threshold for the highpass filter. (Ignored if declump is False)
- `min_size`: The minimum voxel count per declumped seed prior to watershedding. (Ignored if declump is False)

*Note: the [method_declumping](method_declumping.ipynb) can be used to test each of the intermediate steps included in the function below. This may be helpful when determining the appropriate settings to apply here.*

In [26]:
declump = True
dec_sig = 1.0
dec_iter = 1
dec_open = False
dec_adj = 1.0
dec_min_size = 0

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** This code block creates an instance segmentation using the settings provided above.

*In the Napari viewer, the image is added as a "labels" layer where each object appears as a different color.*

In [20]:
from scipy import ndimage
from skimage.morphology import opening

def _highpass_filter(in_img: np.ndarray, sigma:float=0.0,
                     iterations:int=1, open:bool=False) -> np.ndarray:
    """
    Applies a highpass filter to the input image.

    Parameters:
    ---------

    in_img:np.ndarray:
        The input image to apply the filter to
    sigma:float
        The sigma value used for the lowpass filter
    iterations:int
        The number of times to apply the highpass filter
    open:bool
        A true/false statement of whether to apply an opening filter or not
    """
    highpass = in_img.copy().astype(np.float32)
    for _ in range(iterations):
        lowpass = ndimage.gaussian_filter(highpass, sigma)
        highpass -= lowpass
        np.clip(highpass, 0, None, out=highpass)
    if open:
        highpass=opening(highpass)
    return highpass

def _otsu_size_filter(in_img: np.ndarray, thresh_adj:float=1, min_size:int=0) -> np.ndarray:
    """
    Both thresholds the image and ensures that all objects are above specified size

    Parameters:
    ---------
    in_img:np.ndarray
        The input image to apply the filter to
    thresh_adj:float
        A scalar value to adjust the threshold value by
    min_size:int
        The minimum value for the volume of the objects
    
    Returns:
    ----------
    A thresholded np.ndarray
    """
    threshold = threshold_otsu(in_img)
    ots = (in_img >= (threshold*thresh_adj))
    return size_filter(img=ots, min_size=min_size, method='3D')

def watershed_declumping(raw_img:np.ndarray, seg_img:np.ndarray, declump:bool, 
                         sigma:float, iterations:int=1, open:bool=False, 
                         thresh_adj:float=1, min_size:int=0) -> np.ndarray:
    """
    Declumps the input organelle using a highpass and threshold to develop seeds for watershedding

    Parameters:
    ---------
    raw_img:np.ndarray
        The raw image to determine the peak points of intensity gradient
    seg_img:np.ndarray
        The segmentation image to determine the area the organelle is located
    declump:bool
        A true/false statement of whether to declump the organelle or not
    sigma:float
        The sigma value used in the gaussian blur lowpass for the highpass filter
    iterations:int
        The number of times the highpass filter is applied
    open:bool
        A true/false statement of whether to apply an opening filter in the highpass filter
    thresh_adj:float
        A scalar for the otsu threshold
    min_size:int
        A number used to ensure the minimum size of the "seeds" for watershedding
    
    Returns:
    ---------
    A labeled np.ndarray of individual organelles
    """
    seg_img = label(seg_img)
    if declump and iterations>=1:
        highpass = _highpass_filter(in_img=raw_img, sigma=sigma, open=open, iterations=iterations)
        ots = _otsu_size_filter(in_img=highpass, thresh_adj=thresh_adj, min_size=min_size)
        
        return label((seg_img) + watershed(image=(np.max(raw_img)-raw_img), 
                                                      markers=label(ots), 
                                                      mask=seg_img,
                                                      connectivity=np.ones((3, 3, 3), bool)))
    else:
        return seg_img

In [38]:
# create instance segmentation based on connectivity
golgi_labels = watershed_declumping(raw_img = raw_golgi,
                                       seg_img = cleaned_img2,
                                       declump = declump,
                                       sigma = dec_sig,
                                       iterations = dec_iter,
                                       open = dec_open,
                                       thresh_adj = dec_adj,
                                       min_size = dec_min_size)

# adding image to Napari as a new layer
viewer.add_labels(golgi_labels, scale=scale, name=f"8 - Instance segmentation (declump={declump}, sigma={dec_sig}, iterations={dec_iter}, open={dec_open}, thresh_adj={dec_adj}, min_size={dec_min_size})")

<Labels layer '8 - Instance segmentation (declump=True, sigma=1.0, iterations=1, open=False, thresh_adj=1.0, min_size=0) [1]' at 0x1e3d375e230>

In [22]:
### AN ALTERNATIVE IF DECLUMPING IS NOT WORTH IT OR NEEDED ###

# # label each object with a unique integer
# golgi_labels = label(cleaned_img2)

# # adding image to Napari as a new layer
# viewer.add_labels(golgi_labels, scale=scale, name="8 - Instance segmentation")

-----
## **SAVING**

### **`Saving` - Save the segmentation output**

#### &#x1F3C3; **Run code; no user input required**
&#x1F453; **FYI:** This code block saves the instance segmentation output to the `out_data_path` specified earlier.

In [23]:
# Saving file
# out_file_n = export_inferred_organelle(golgi_labels, "golgi", meta_dict, out_data_path)
# print(f"saved to: {out_data_path}")

-----
-----
## **Define `infer_golgi` function**

The following code includes an example of how the workflow steps above are combined into one function. This function can be run below to process a single image. It is included in the [batch process notebook](batch_process_segmentations.ipynb) to run the above analysis on multiple cells. 

This function can be utilized from infer-subc using:
```python
infer_subc.organelles.golgi.infer_golgi()
```

#### &#x1F3C3; **Run code; no user input required**
&#x1F453; **FYI:** This code block defines the `infer_golgi()` function. It is applied below.

In [37]:
##########################
#  infer_golgi
##########################
def _infer_golgi(
            in_img: np.ndarray,
            golgi_ch: int,
            median_sz: int,
            gauss_sig: float,
            mo_method: str,
            mo_adjust: float,
            mo_cutoff_size: int,
            min_thickness: int,
            thin_dist: int,
            dot_scale_1: float,
            dot_cut_1: float,
            dot_scale_2: float,
            dot_cut_2: float,
            dot_scale_3: float,
            dot_cut_3: float,
            dot_method: str,
            min_hole_w: int,
            max_hole_w: int,
            small_obj_w: int,
            fill_filter_method: str,
                declump: bool,
                dec_sig: float,
                dec_iter: int,
                dec_open: bool,
                dec_adj: float,
                dec_min_size: int
        ) -> np.ndarray:

    """
    Procedure to infer golgi from linearly unmixed input.

   Parameters
    ------------
    in_img: 
        a 3d image containing all the channels
    median_sz: 
        width of median filter for signal
    mo_method: 
         which method to use for calculating global threshold. Options include:
         "triangle" (or "tri"), "median" (or "med"), and "ave_tri_med" (or "ave").
         "ave" refers the average of "triangle" threshold and "mean" threshold.
    mo_adjust: 
        Masked Object threshold `local_adjust`
    mo_cutoff_size: 
        Masked Object threshold `size_min`
    min_thinkness: 
        Half of the minimum width you want to keep from being thinned.
        For example, when the object width is smaller than 4, you don't
        want to make this part even thinner (may break the thin object
        and alter the topology), you can set this value as 2.
    thin_dist: 
        the amount to thin (has to be an positive integer). The number of
         pixels to be removed from outter boundary towards center.
    dot_scale: 
        scales (log_sigma) for dot filter (1,2, and 3)
    dot_cut: 
        threshold for dot filter thresholds (1,2,and 3)
    small_obj_w: 
        minimu object size cutoff for nuclei post-processing
    
    Returns
    -------------
    golgi_object
        mask defined extent of golgi object
    """

    ###################
    # EXTRACT
    ###################    
    golgi = select_channel_from_raw(in_img, golgi_ch)

    ###################
    # PRE_PROCESSING
    ###################    
    golgi_smooth =  scale_and_smooth(golgi,
                              median_size = median_sz, 
                              gauss_sigma = gauss_sig)
    ###################
    # CORE_PROCESSING
    ###################
    bw = masked_object_thresh(golgi_smooth, global_method=mo_method, cutoff_size=mo_cutoff_size, local_adjust=mo_adjust)

    bw_thin = topology_preserving_thinning(bw, min_thickness, thin_dist)

    bw_extra = dot_filter_3(golgi_smooth, dot_scale_1, dot_cut_1, dot_scale_2, dot_cut_2, dot_scale_3, dot_cut_3, dot_method)

    bw = np.logical_or(bw_extra, bw_thin)
    ###################
    # POST_PROCESSING
    ###################
    struct_obj = fill_and_filter_linear_size(bw, 
                                             hole_min=min_hole_w, 
                                             hole_max=max_hole_w, 
                                             min_size=small_obj_w,
                                             method=fill_filter_method)

    ###################
    # LABELING
    ###################
    struct_obj1 = watershed_declumping(raw_img = golgi,
                                       seg_img = struct_obj,
                                       declump = declump,
                                       sigma = dec_sig,
                                       iterations = dec_iter,
                                       open = dec_open,
                                       thresh_adj = dec_adj,
                                       min_size = dec_min_size)

    return struct_obj1


#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** This code block applies the function above to your test image. The settings specified above are applied here.

In [39]:
_golgi_object =  _infer_golgi(
                                img_data,
                                GOLGI_CH,
                                median_sz,
                                gauss_sig,
                                thresh_method,
                                thresh_adj,
                                cell_wise_min_area,
                                thin_dist_preserve,
                                thin_dist,
                                dot_scale_1,
                                dot_cut_1,
                                dot_scale_2,
                                dot_cut_2,
                                dot_scale_3,
                                dot_cut_3,
                                dot_method,
                                hole_min_width,
                                hole_max_width,
                                small_object_width,
                                fill_filter_method,
                                    declump,
                                    dec_sig,
                                    dec_iter,
                                    dec_open,
                                    dec_adj,
                                    dec_min_size) 

#confirm this output matches the output saved above
print(f"The segmentation output here matches the output created above: {np.all(golgi_labels == _golgi_object)}")

The segmentation output here matches the output created above: True


In [40]:
viewer.add_labels(_golgi_object, scale=scale, name=f"9 - Golgi Object from infer_golgi function (declump={declump}, sigma={dec_sig}, iterations={dec_iter}, open={dec_open}, thresh_adj={dec_adj}, min_size={dec_min_size})")

<Labels layer '9 - Golgi Object from infer_golgi function (declump=True, sigma=1.0, iterations=1, open=False, thresh_adj=1.0, min_size=0)' at 0x1e3cab5cd00>

-------------
### ✅ **INFER GOLGI COMPLETE!**

Continue on to other `organelle` notebooks as needed:
- Infer [`lysosomes`](1.2_infer_lysosome.ipynb)
- Infer [`mitochondria`](1.3_infer_mitochondria.ipynb)
- Infer [`peroxisomes`](1.5_infer_peroxisome.ipynb)
- Infer [`endoplasmic reticulum (ER)`](1.6_infer_ER.ipynb)
- Infer [`lipid droplets`](1.7_infer_lipid_droplet.ipynb)

Alternatively, proceed to **one** of the following `mask` segmentation notebooks as needed:
| **Imaging Requirements**|[**Masks Workflow**](./1.1_infer_masks_from-composite_with_nuc.ipynb)|[**Masks Workflow (A)**](./1.1a_infer_masks_from-composite_single_cell.ipynb)|[**Masks Workflow (B)**](./1.1b_infer_masks_from-composite_multiple-cells.ipynb)|[**Masks Workflow (C)**](./1.1c_infer_masks_from-composite_neuron_with_pm.ipynb)|[**Masks Workflow (D)**](./1.1d_infer_masks_from-composite_ipsc.ipynb)|
| :------------------------------------- |  :------:  |  :------:  |  :------:  |  :------:  |  :------:  |
| Nuclei Marker                          |     ✔     |      ✘     |     ✘     |     ✔     |     ✔     |
| Cell Membrane Marker                   |     ✘     |      ✘     |     ✘     |     ✔     |     ✔     |
| Cytoplasmic Organelles                 |     ✔     |      ✔     |     ✔     |     ✔     |     ✔     |
| Number of cells per image              |  Single or Multiple  |   Single   |  Single or Multiple |  Single or Multiple |  Single or Multiple |
| Applicable with sample data         |  ✘  |   neuron_1   |  astrocyte | neuron_2| iPSC|

Or proceed to batch processing here: [batch process notebook](batch_process_segmentations.ipynb)